# Tests and Evals: From One Example to an Evaluation Dataset

One example is not enough. This notebook shows why and how to build a repeatable evaluation suite using structured output.

In [1]:
import os
import pandas as pd
from typing import Literal
from pydantic import BaseModel
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

In [2]:
load_dotenv()

True

In [3]:
client = OpenAI()

In [4]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

In [5]:
with open("system_prompt_v1.md") as f:
    PROMPT_V1 = f.read()

In [6]:
demand = pd.read_csv("demand_signals.csv")
display(demand)

,region,property_type,bedrooms,search_volume,lead_count,recent_sales,demand_score
0,west,apartment,1,890,64,24,0.77
1,west,apartment,2,1240,86,34,0.91
2,west,house,3,520,38,14,0.63
3,west,studio,1,720,51,19,0.69
4,north,apartment,1,620,31,12,0.54
5,north,apartment,2,940,71,26,0.80
6,north,house,2,310,22,8,0.41
7,north,house,3,280,19,6,0.37
8,south,apartment,2,780,58,19,0.72
9,south,house,3,710,42,18,0.68


In [7]:
supply = pd.read_csv("supply_snapshot.csv")
display(supply)

,region,property_type,bedrooms,active_listings,avg_days_on_market,supply_score
0,west,apartment,1,42,31,0.44
1,west,apartment,2,18,21,0.22
2,west,house,3,67,48,0.72
3,west,studio,1,48,36,0.52
4,north,apartment,1,77,54,0.81
5,north,apartment,2,24,28,0.31
6,north,house,2,88,62,0.88
7,north,house,3,94,68,0.91
8,south,apartment,2,39,35,0.48
9,south,house,3,36,39,0.55


In [8]:
scenarios = pd.read_csv("eval_golden_dataset.csv")
display(scenarios.head())

,scenario_id,region,property_type,bedrooms,expected_priority,expected_action,rationale
0,G001,west,apartment,2,high,prioritize_new_listings,demand=0.91 supply=0.22 — strongest demand in ...
1,G002,north,apartment,1,low,deprioritize,demand=0.54 supply=0.81 — moderate demand but ...
2,G003,south,house,3,medium,monitor_opportunity,demand=0.68 supply=0.55 — balanced signals wit...
3,G004,east,apartment,2,high,prioritize_new_listings,demand=0.82 supply=0.37 — clear high demand wi...
4,G005,center,studio,1,low,deprioritize,demand=0.47 supply=0.88 — weak demand and seve...


## Part 1: The One-Example Problem

v0.1 returned free-form text. Let's run one example and see what we get.

In [9]:
def build_context(region: str, property_type: str, bedrooms: int) -> str:
    d = demand[
        (demand["region"] == region)
        & (demand["property_type"] == property_type)
        & (demand["bedrooms"] == bedrooms)
    ]
    s = supply[
        (supply["region"] == region)
        & (supply["property_type"] == property_type)
        & (supply["bedrooms"] == bedrooms)
    ]
    if d.empty or s.empty:
        return "No data available for the requested segment."
    return (
        f"Region: {region}\n"
        f"Property: {bedrooms}-bedroom {property_type}\n\n"
        f"Demand signals:\n{d.to_string(index=False)}\n\n"
        f"Supply snapshot:\n{s.to_string(index=False)}"
    )

In [10]:
def ask_realthor_v1(region: str, property_type: str, bedrooms: int, prompt: str) -> str:
    context = build_context(region, property_type, bedrooms)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.2,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": context},
        ],
    )
    return response.choices[0].message.content

In [11]:
one_example = ask_realthor_v1("west", "apartment", 2, PROMPT_V1)
print(one_example)

- **Priority**: High
- **Reasoning based on demand**: The demand signals indicate a strong interest in 2-bedroom apartments in the west region, with a search volume of 1,240 and a lead count of 86. The demand score of 0.91 suggests that there is a high level of interest and potential for conversion.
- **Reasoning based on supply**: In contrast, the supply snapshot shows only 18 active listings for 2-bedroom apartments in the same region, with an average of 21 days on the market. The supply score of 0.22 indicates that the current supply is significantly low compared to the demand.
- **Recommended action**: The company should prioritize bringing more 2-bedroom apartments to the platform in the west region to capitalize on the high demand and low supply.
- **Confidence**: High


Looks good. But how do we know it's actually correct? And will it be consistent?

Let's try to evaluate it with string matching.

In [12]:
def check_priority_string(output: str, expected_priority: str) -> bool:
    return expected_priority in output.lower()

In [13]:
# This passes — but notice that "high" also appears in "high supply", "high confidence", etc.
# A "low" priority scenario could pass this check if the model says "high supply" anywhere.
print(check_priority_string(one_example, "high"))

True


String matching is unreliable. The word "high" can appear in many contexts. We need a better approach.

## Part 2: Structured Output Makes Evaluation Deterministic

With `with_structured_output()`, the model returns a validated Pydantic object. The priority field is either `"high"`, `"medium"`, or `"low"` — nothing else.

In [14]:
class ListingRecommendation(BaseModel):
    """Structured recommendation from RealThor."""
    priority: Literal["high", "medium", "low"]
    demand_reasoning: str
    supply_reasoning: str
    recommended_action: Literal["prioritize_new_listings", "monitor_opportunity", "deprioritize"]
    confidence: Literal["high", "medium", "low"]

In [15]:
structured_llm = llm.with_structured_output(ListingRecommendation)

In [16]:
def ask_realthor_v2(region: str, property_type: str, bedrooms: int, prompt: str) -> ListingRecommendation:
    context = build_context(region, property_type, bedrooms)
    return structured_llm.invoke([
        SystemMessage(content=prompt),
        HumanMessage(content=context),
    ])

In [17]:
def evaluate_output(result: ListingRecommendation, row) -> dict:
    priority_match = result.priority == row["expected_priority"]
    action_match = result.recommended_action == row["expected_action"]
    passed = priority_match and action_match
    failure_reason = ""
    if not priority_match:
        failure_reason += f"priority: got '{result.priority}', expected '{row['expected_priority']}'; "
    if not action_match:
        failure_reason += f"action: got '{result.recommended_action}', expected '{row['expected_action']}'"
    return {
        "passed": passed,
        "priority": result.priority,
        "action": result.recommended_action,
        "failure_reason": failure_reason.strip("; "),
    }

## Part 3: Evaluation Across All 20 Scenarios (v1 prompt)

In [18]:
results = []

for _, row in scenarios.iterrows():
    result = ask_realthor_v2(row["region"], row["property_type"], row["bedrooms"], PROMPT_V1)
    eval_result = evaluate_output(result, row)
    results.append({
        "scenario_id": row["scenario_id"],
        "region": row["region"],
        "property_type": row["property_type"],
        "bedrooms": row["bedrooms"],
        "expected_priority": row["expected_priority"],
        "expected_action": row["expected_action"],
        **eval_result,
    })

In [19]:
results_df = pd.DataFrame(results)
display(results_df)

,scenario_id,region,property_type,bedrooms,expected_priority,expected_action,passed,priority,action,failure_reason
0,G001,west,apartment,2,high,prioritize_new_listings,True,high,prioritize_new_listings,
1,G002,north,apartment,1,low,deprioritize,False,high,prioritize_new_listings,"priority: got 'high', expected 'low'; action: ..."
2,G003,south,house,3,medium,monitor_opportunity,False,high,prioritize_new_listings,"priority: got 'high', expected 'medium'; actio..."
3,G004,east,apartment,2,high,prioritize_new_listings,True,high,prioritize_new_listings,
4,G005,center,studio,1,low,deprioritize,False,medium,monitor_opportunity,"priority: got 'medium', expected 'low'; action..."
5,G006,west,apartment,1,high,prioritize_new_listings,True,high,prioritize_new_listings,
6,G007,west,house,3,low,deprioritize,False,medium,monitor_opportunity,"priority: got 'medium', expected 'low'; action..."
7,G008,north,apartment,2,high,prioritize_new_listings,True,high,prioritize_new_listings,
8,G009,north,house,2,low,deprioritize,False,high,prioritize_new_listings,"priority: got 'high', expected 'low'; action: ..."
9,G010,south,apartment,2,medium,monitor_opportunity,False,high,prioritize_new_listings,"priority: got 'high', expected 'medium'; actio..."


In [20]:
passed = results_df["passed"].sum()
total = len(results_df)
print(f"v1 pass rate: {passed}/{total} ({passed/total:.0%})")
print()
failures = results_df[~results_df["passed"]]
if not failures.empty:
    print("Failures:")
    display(failures[["scenario_id", "region", "property_type", "bedrooms", "expected_priority", "priority", "failure_reason"]])

v1 pass rate: 6/20 (30%)

Failures:


,scenario_id,region,property_type,bedrooms,expected_priority,priority,failure_reason
1,G002,north,apartment,1,low,high,"priority: got 'high', expected 'low'; action: ..."
2,G003,south,house,3,medium,high,"priority: got 'high', expected 'medium'; actio..."
4,G005,center,studio,1,low,medium,"priority: got 'medium', expected 'low'; action..."
6,G007,west,house,3,low,medium,"priority: got 'medium', expected 'low'; action..."
8,G009,north,house,2,low,high,"priority: got 'high', expected 'low'; action: ..."
9,G010,south,apartment,2,medium,high,"priority: got 'high', expected 'medium'; actio..."
10,G011,south,house,4,low,medium,"priority: got 'medium', expected 'low'; action..."
11,G012,east,studio,1,low,high,"priority: got 'high', expected 'low'; action: ..."
12,G013,east,house,3,low,medium,"priority: got 'medium', expected 'low'; action..."
13,G014,center,apartment,1,low,high,"priority: got 'high', expected 'low'; action: ..."
